# 6v6 双方对攻 · 23×12 米

普通 CPU，运行全部后点击播放。每人两命；开局弓在手、箭在中线，每人分配三支（全场 36 支）。必须到中线再跑回己方底线才解锁射击。每半场两个掩体。保留之前的箭速、η 映射、射击距离、移动速度和复活参数。

暂定胜负：一方全员耗尽两命，另一方胜；120 秒结束仍有双方剩余命数，或同一 tick 双方全灭，则平局。新模式不设靶架。双方不能越过中线，中心坐标须留出玩家身体半径。所有脚本和未实测参数仍待校准。


In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT_DIR = Path("/content/archery-tag-simulate")
REPO_URL = "https://github.com/Echoudexigaigu/archery-tag-simulate.git"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    if not (PROJECT_DIR / ".git").exists():
        raise RuntimeError("项目目录没有 Git 历史，请先备份，再使用一个新的 Colab 运行时。")
    remote = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "remote", "get-url", "origin"], text=True).strip()
    if remote.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("已有目录属于其他仓库；请使用新的运行时。")
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True).strip()
    if dirty:
        print("保留已有未提交改动，不自动更新。请先保存，再决定如何更新。")
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
if not (PROJECT_DIR / "src/archery_tag_balance/environment.py").is_file():
    raise RuntimeError("已有目录还是旧骨架。保存改动后更新仓库，或在新运行时重新运行。")
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True).strip())

# Reload the updated package even when this Colab runtime previously ran attack/defense.
import importlib
for module_name in list(sys.modules):
    if module_name == "archery_tag_balance" or module_name.startswith("archery_tag_balance."):
        del sys.modules[module_name]
importlib.invalidate_caches()
if not (PROJECT_DIR / "src/archery_tag_balance/skirmish.py").is_file():
    raise RuntimeError("仓库未更新到6v6 对攻版本；请保存本地改动后更新，或使用新运行时。")


## 阵容与配置

`attack` 对应红队，`defense` 对应蓝队，仅保留原有接口键名；两边使用同一种对攻策略。默认仍每队 2 名老手，个体 η 范围保持原值。初始箭按个人配额放在中线，取走后落地或射出的箭可以在本方半场回收。

第一条命耗尽：3 秒后在底线空箭复活，0.8 秒保护，越保护线或捡箭提前失效。完成过开局往返的人复活后不必重复往返；未完成者不能通过复活传送绕过这一要求。第二条命耗尽则永久出局；本模式没有接箭或打靶复活。


In [ ]:
import json
from archery_tag_balance import SimulationEngine
from archery_tag_balance.experiments import save_json, source_digest
config = json.loads((PROJECT_DIR / "configs/skirmish_6v6_23x12.json").read_text())
SEED = 1000
config["teams"]["attack"]["veterans"] = 2
config["teams"]["defense"]["veterans"] = 2
print("掩体坐标:", [o["center_xy_m"] for o in config["obstacles"]])
print("中线 x =", config["opening"]["center_x_m"], "米")


## 运行单局

只记录全局回放；脚本决策仍使用视线受限的局部观测。


In [ ]:
summary, env = SimulationEngine(config).episode(
    seed=SEED, record=True, record_interval_s=0.1, check_invariants=True
)
print(json.dumps({k:v for k,v in summary.items() if k not in ("eta", "resource_curve")}, ensure_ascii=False, indent=2))
run_dir = PROJECT_DIR / "runs" / f"skirmish_{SEED}"
save_json(run_dir / "config.json", config)
save_json(run_dir / "episode.json", {"summary":summary,"trace":env.trace,"events":env.events,"source_sha256":source_digest(),"git_commit":subprocess.check_output(["git","-C",str(PROJECT_DIR),"rev-parse","HEAD"],text=True).strip()})


## 播放与保存

红队、蓝队、剩余总命数、个人持箭和射击解锁状态随时间更新。支持逐帧、拖动、暂停、倍速。黄色箭头为飞行箭的俯视投影。


In [ ]:
from IPython.display import HTML, display
from archery_tag_balance.replay import replay_html
html = replay_html(config, env, speed=2.0)
display(HTML(html))
replay_path = run_dir / "replay.html"
replay_path.write_text(html, encoding="utf-8")
print("HTML 回放:", replay_path)


## 小批量检查

默认 20 局只用于检查流程，不足以证明现实规则平衡。统计分别给出红胜、蓝胜和平局；`attack_win_rate_given_decisive` 是剔除平局后的红方胜率。大量平局不能解释成平衡。改变 N 或老手人数可以继续比较。


In [ ]:
N = 20
stats, episodes = SimulationEngine(config).batch(range(2000, 2000 + N), keep_curves=True)
print(json.dumps(stats, ensure_ascii=False, indent=2))
save_json(run_dir / "batch.json", {"config":config,"summary":stats,"episodes":episodes,"source_sha256":source_digest()})


## 可选下载

在新代码格执行以下代码下载回放。Colab 运行时文件是临时的，跨会话保留需要下载或自行备份到 Drive。
```python
from google.colab import files
files.download(str(replay_path))
```
